In [ ]:
import os

SEED = 0 


os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8") 


# -------------------------------------------------------------
# now import libraries
import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

# Torch
import torch
torch.manual_seed(SEED)


import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import logging
import triku as tk 
from matplotlib import pylab
import os
import sys
import yaml
from scipy.sparse import csr_matrix
import gc
import torch
import scanit
from scipy.sparse import issparse
import scanit
from scipy.sparse import issparse
import gc
import torch

In [ ]:
import rapids_singlecell as rsc

In [ ]:
rsc.__version__

In [ ]:
nThreads = 10
import os

os.environ["OMP_NUM_THREADS"] = f"{nThreads}"
os.environ["OPENBLAS_NUM_THREADS"] = f"{nThreads}"
os.environ["MKL_NUM_THREADS"] = f"{nThreads}"
os.environ["BLIS_NUM_THREADS"] = f"{nThreads}"
os.environ["VECLIB_MAXIMUM_THREADS"] = f"{nThreads}"
os.environ["MKL_DYNAMIC"] = "FALSE"


In [ ]:

homeDir = os.getenv("HOME")

sys.path.insert(1, homeDir+"/utils/")


from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _DEGs_utils import *

from _Aggregation import *
from _plotting import *
import rapids_singlecell as rsc

import ipynbname
import nbconvert.exporters
from nbconvert.preprocessors import TagRemovePreprocessor
import os


try:
    nb_name = ipynbname.name()
except:
    nb_name = "".join(os.path.basename(globals()['__vsc_ipynb_file__']))

print(nb_name)


In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)
DS = "B19-25653_8um_banksy"
FigTag = "B19-25653"
DSname = "Banksy_B19-25653_8um"

base_path = "/data/Spatial_Tx" 
HashesDir = homeDir+"/hashes"

import cupyx.scipy.sparse
import random
from scipy import sparse
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import cupy as cp

cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
%load_ext rpy2.ipython
pd.DataFrame.iteritems = pd.DataFrame.items

# Load data

In [ ]:
adata = sc.read_h5ad(f"/data/projects/spatialTX/3_Domain_characterization/{DS}_partitioned.h5ad")
adata = adata[adata.obs.spot_class.isin(["doublet_certain","singlet"])].copy()
adata


# We keep certain doublets only if above selected % of total spots

In [ ]:
minRate = 0.05
adata = adata[adata.obs["spacexr"] != "reject"].copy()
adata = adata[(~adata.obs["spacexr"].str.contains(",")) | (adata.obs["spacexr"].str.contains(",") & adata.obs["spacexr"].isin(adata.obs["spacexr"].value_counts()[adata.obs["spacexr"].value_counts() > adata.shape[0]*minRate].index.tolist()))].copy()

print(adata.shape)



adata.obs["Myeloids_vs_Niche"] = np.nan
adata.obs["Myeloids_vs_Niche"] = np.where(adata.obs["spacexr"] == "Myeloids", "Myeloids","Niche")
adata.obs["Myeloids_vs_Niche"] = "Domain" + adata.obs["AnnotatedDomain"].astype(str) +  "_"  +adata.obs["Myeloids_vs_Niche"].astype(str)




In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

counts = adata.obs["Myeloids_vs_Niche"].value_counts(dropna=False)  # include NaN if you want
df = counts.rename_axis("CellType").reset_index(name="Count")
df = df.sort_values("Count", ascending=False)

n_total = df["Count"].sum()
df["Frac"] = df["Count"] / n_total

order = df["CellType"].tolist()

fig, ax = plt.subplots(figsize=(20, 6))
sns.barplot(data=df, x="CellType", y="Count", order=order, color="steelblue", ax=ax)

ymax = df["Count"].max()
ax.set_ylim(0, ymax * 1.12)

for patch, frac in zip(ax.patches, df["Frac"].to_numpy()):
    h = patch.get_height()
    ax.annotate(
        f"{frac:.1%}",
        (patch.get_x() + patch.get_width() / 2, h),
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=90,
        xytext=(0, 2),
        textcoords="offset points",
    )

ax.set_ylabel("Number of cells")
ax.set_title("Cell counts by spacexR deconvolution")
ax.tick_params(axis="x", rotation=90)
sns.despine(offset=10)

fig.tight_layout()
plt.show()


In [ ]:
adata.obs["Myeloids_vs_Niche"].value_counts()

# 1) Myeloids

In [ ]:
Celltype = "Myeloids"



In [ ]:
adataNiches = adata.copy()
adataNiches.obs["contrastCol"] = adataNiches.obs["Myeloids_vs_Niche"].astype(str)
print(adataNiches.obs["contrastCol"].value_counts())

In [ ]:
minCells = adataNiches.shape[1]
minCells = 100

SelectedDomains = adataNiches.obs["contrastCol"].value_counts()[adataNiches.obs["contrastCol"].value_counts() > minCells].index.tolist()
print(SelectedDomains)
adataNiches = adataNiches[adataNiches.obs["contrastCol"].isin(SelectedDomains)].copy()

In [ ]:


MatchedNiches = adataNiches.obs.loc[adataNiches.obs["contrastCol"].str.contains(Celltype),"AnnotatedDomain"].unique().tolist()
adataNiches = adataNiches[adataNiches.obs["AnnotatedDomain"].isin(MatchedNiches)].copy()


adataNiches.X = adataNiches.layers["counts"].copy()



assign_palette_topn_then_random(
    adataNiches,
    obs_key="contrastCol",
    palette_name="Dark2",   # try "tab10", "Set3", "Accent", etc.
    random_seed=123
)


# Store initial mappings
colDictGenotype = dict(zip([i for i in adataNiches.obs["contrastCol"].cat.categories.tolist()], 
		adataNiches.uns["contrastCol_colors"]))

colDictGenotype



celltype_colors = dict(zip( adataNiches.obs["contrastCol"].cat.categories , adataNiches.uns["contrastCol_colors"]))
celltype_colors

In [ ]:
sc.set_figure_params(dpi=100, facecolor='white', dpi_save=500)
for niche in adataNiches.obs["AnnotatedDomain"].unique():
    print(niche)
    localAdata = adataNiches[adataNiches.obs["AnnotatedDomain"] == niche]
    assign_palette_topn_then_random(
        localAdata,
        obs_key="contrastCol",
        palette_name="Set2",   # try "tab10", "Set3", "Accent", etc.
        random_seed=123
    )
    print("plotting")
    plot_spatial_obs(
        localAdata, obs="contrastCol", width=7, dpi=150,dotscale=1.5 ,img_key="hires",marker='o',
        spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", enforce_pixel_spot_size=False, img_alpha=.4,  legend_kwargs={"fontsize":10})
    del localAdata

In [ ]:
assign_palette_topn_then_random(
    adataNiches,
    obs_key="contrastCol",
    palette_name="Paired",   # try "tab10", "Set3", "Accent", etc.
    random_seed=123
)

assign_palette_topn_then_random(
    adataNiches,
    obs_key="spacexr",
    palette_name="Paired",   # try "tab10", "Set3", "Accent", etc.
    random_seed=123
)


fig, ax, info = plot_stacked_fractions_by_cluster(
   adataNiches,
    variable1="spacexr",
    variable2="contrastCol",
    variable3="AnnotatedDomain",
    aggregate_by_variable3=False,figsize=(30,5),linewidth=0, annotations=["AnnotatedDomain"], legend_hspace=1,legend_row_height=10,legend_cols_annotations=3,legend_fontsize=10,legend_title_fontsize=15,
    renormalize_medians=False,
)

In [ ]:
new_colors = {
    "2": "#63000B",
    "3": "#4BC9BE",
}
cats = list(adata.obs["Spatial_Domain"].cat.categories)

ca3ts = list(adata.obs["Spatial_Domain"].cat.categories)
old_map = dict(zip(cats, adata.uns["Spatial_Domain_colors"]))

old_map.update(new_colors)

adata.uns["Spatial_Domain_colors"] = [old_map[c] for c in cats]

In [ ]:

plot_spatial_obs(
    adata[adata.obs["spacexr"].str.contains("Myeloids")], obs="Spatial_Domain", width=7, dpi=150,dotscale=1.5 ,img_key="hires",marker='o',groups=["2","3"],
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", enforce_pixel_spot_size=False, img_alpha=.4,  legend_kwargs={"fontsize":10})

# Plot myeloids per domain

In [ ]:

plot_spatial_obs(
    adataNiches[~adataNiches.obs["contrastCol"].str.contains("_Niche")], obs="contrastCol", width=10, dpi=150,dotscale=2 ,img_key="hires",marker='o',linewidths=.06,
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", enforce_pixel_spot_size=False, img_alpha=.4,  legend_kwargs={"fontsize":10})


In [ ]:
import itertools
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform


def _get_expression_matrix(adata, genes: List[str], layer: Optional[str] = None):
    """
    Return expression matrix for the selected genes from adata.X or adata.layers[layer].
    """
    adata_sub = adata[:, genes]

    if layer is None:
        X = adata_sub.X
    else:
        if layer not in adata_sub.layers:
            raise KeyError(f"Layer '{layer}' not found in adata.layers")
        X = adata_sub.layers[layer]

    return X


def pick_top_first_owner(
    degs_sub: pd.DataFrame,
    top_n: int,
    deg_group_name: str,
    first_owner: Dict[str, str],
    allowed_genes: set,
    gene_col: str = "gene",
    fdr_col: str = "FDR",
    logfc_col: str = "logFC",
    min_logfc: float = 1.0,
    fdr_thresh: float = 0.05,
    sort_by: str = "FDR",
    ascending: bool = True,
) -> List[str]:
    """
    Pick up to top_n marker genes for one DEG group, enforcing global first-owner uniqueness.

    Parameters
    ----------
    sort_by
        Column used to rank candidate genes before first-owner assignment.
    ascending
        Whether sorting is ascending for sort_by.
    """
    df2 = degs_sub.loc[
        (degs_sub[fdr_col] < fdr_thresh) & (degs_sub[logfc_col] > min_logfc)
    ].copy()

    if sort_by not in df2.columns:
        raise KeyError(f"sort_by column '{sort_by}' not found in DEG dataframe")

    df2 = df2.sort_values(sort_by, ascending=ascending)

    picked = []
    for gene in df2[gene_col].astype(str):
        if gene not in allowed_genes:
            continue
        if gene in first_owner:
            continue

        first_owner[gene] = deg_group_name
        picked.append(gene)

        if len(picked) >= top_n:
            break

    return picked

def select_unique_markers_from_deg_df(
    degs_df: pd.DataFrame,
    adata,
    deg_group_col: str,
    gene_col: str = "gene",
    top_n: int = 10,
    fdr_col: str = "FDR",
    logfc_col: str = "logFC",
    min_logfc: float = 1.0,
    fdr_thresh: float = 0.05,
    deg_group_order: Optional[List[str]] = None,
    sort_by: str = "FDR",
    ascending: bool = True,
    verbose: bool = True,
) -> Tuple[Dict[str, List[str]], Dict[str, str], List[str]]:
    """
    Select globally unique markers from one long DEG dataframe.
    """
    allowed_genes = set(adata.var_names.astype(str))
    first_owner: Dict[str, str] = {}
    markers_dict: Dict[str, List[str]] = {}

    if deg_group_order is None:
        deg_group_order = degs_df[deg_group_col].drop_duplicates().tolist()

    for deg_group_name in deg_group_order:
        degs_sub = degs_df.loc[degs_df[deg_group_col] == deg_group_name].copy()

        markers = pick_top_first_owner(
            degs_sub=degs_sub,
            top_n=top_n,
            deg_group_name=deg_group_name,
            first_owner=first_owner,
            allowed_genes=allowed_genes,
            gene_col=gene_col,
            fdr_col=fdr_col,
            logfc_col=logfc_col,
            min_logfc=min_logfc,
            fdr_thresh=fdr_thresh,
            sort_by=sort_by,
            ascending=ascending,
        )
        markers_dict[deg_group_name] = markers

        if verbose:
            print(f"{deg_group_name} -> {len(markers)} markers")

    unique_markers = list(first_owner.keys())
    gene_to_deg_group = first_owner.copy()

    return markers_dict, gene_to_deg_group, unique_markers


def compute_marker_group_stats(
    adata,
    markers_dict: Dict[str, List[str]],
    obs_groupby: str,
    marker_group_label: str = "marker_group",
    layer: Optional[str] = None,
    positive_threshold: float = 0,
) -> pd.DataFrame:
    """
    Compute positive rate and mean expression for selected markers per observation group.

    Parameters
    ----------
    adata
        AnnData object
    markers_dict
        Dict mapping marker-group -> list of selected genes
    obs_groupby
        Column in adata.obs defining the groups across which to summarize expression
    marker_group_label
        Name of output column that stores which DEG group originally owned the marker
    layer
        If None, use adata.X. Otherwise use adata.layers[layer].
    positive_threshold
        Positive rate is computed as fraction of cells with expression > positive_threshold.
        For sparse matrices, exact zero-threshold logic is only used when positive_threshold == 0.
    """
    obs_groups = adata.obs[obs_groupby].astype("category")
    groups = list(obs_groups.cat.categories)

    obs_vals = obs_groups.to_numpy()
    group_indices = {g: np.where(obs_vals == g)[0] for g in groups}

    stats_list = []

    for marker_group, markers in markers_dict.items():
        if not markers:
            continue

        X = _get_expression_matrix(adata, markers, layer=layer)
        if sparse.issparse(X):
            X = X.tocsr()
        else:
            X = np.asarray(X)

        for obs_group, idx in group_indices.items():
            if idx.size == 0:
                continue

            Xg = X[idx, :]
            n_cells = idx.size

            if sparse.issparse(Xg):
                mean_expr = np.asarray(Xg.mean(axis=0)).ravel()

                if positive_threshold == 0:
                    pos_rate = np.asarray(Xg.getnnz(axis=0)).ravel() / n_cells
                else:
                    pos_rate = np.asarray((Xg > positive_threshold).mean(axis=0)).ravel()
            else:
                mean_expr = np.asarray(Xg.mean(axis=0)).ravel()
                pos_rate = np.asarray((Xg > positive_threshold).mean(axis=0)).ravel()

            stats_list.append(
                pd.DataFrame(
                    {
                        obs_groupby: obs_group,
                        "gene": markers,
                        "positive_rate": pos_rate,
                        "mean_expression": mean_expr,
                        "n_cells": n_cells,
                        marker_group_label: marker_group,
                    }
                )
            )

    if not stats_list:
        return pd.DataFrame(
            columns=[
                obs_groupby,
                "gene",
                "positive_rate",
                "mean_expression",
                "n_cells",
                marker_group_label,
            ]
        )

    return pd.concat(stats_list, ignore_index=True)


def compute_group_average_expression(
    adata,
    genes: List[str],
    obs_groupby: str,
    layer: Optional[str] = None,
) -> pd.DataFrame:
    """
    Compute average expression per gene per observation group using adata.X or a selected layer.

    Returns
    -------
    DataFrame with rows = genes, columns = observation groups
    """
    obs_order = adata.obs[obs_groupby].drop_duplicates().tolist()
    out = []

    for group in obs_order:
        idx = np.where(adata.obs[obs_groupby].to_numpy() == group)[0]
        X = _get_expression_matrix(adata, genes, layer=layer)

        if sparse.issparse(X):
            vals = np.asarray(X[idx, :].mean(axis=0)).ravel()
        else:
            vals = np.asarray(X[idx, :]).mean(axis=0)

        out.append(pd.Series(vals, index=genes, name=group))

    avg_expr = pd.concat(out, axis=1)
    return avg_expr


def order_groups_by_correlation(
    avg_expr: pd.DataFrame,
    method: str = "average",
) -> Tuple[List[str], pd.DataFrame]:
    """
    Order observation groups by hierarchical clustering on correlation distance.
    """
    corr = avg_expr.corr()
    dist = 1 - corr.clip(-1, 1)

    Z = linkage(squareform(dist.values, checks=False), method=method)
    ordered_groups = corr.index[leaves_list(Z)].to_list()

    return ordered_groups, corr


def order_genes_by_dominant_group(
    avg_expr: pd.DataFrame,
    ordered_groups: List[str],
) -> List[str]:
    """
    Order genes by the observation group in which they have maximal mean expression.
    """
    gene_to_max_group = avg_expr.idxmax(axis=1)

    ordered_genes = list(
        itertools.chain.from_iterable(
            gene_to_max_group[gene_to_max_group == group].index.tolist()
            for group in ordered_groups
        )
    )

    return ordered_genes


def run_marker_selection_and_ordering(
    adata,
    degs_df: pd.DataFrame,
    deg_group_col: str,
    obs_groupby: str,
    gene_col: str = "gene",
    top_n: int = 10,
    fdr_col: str = "FDR",
    logfc_col: str = "logFC",
    min_logfc: float = 1.0,
    fdr_thresh: float = 0.05,
    deg_group_order: Optional[List[str]] = None,
    marker_group_label: str = "marker_group",
    layer: Optional[str] = None,
    positive_threshold: float = 0,
    sort_by: str = "FDR",
    ascending: bool = True,
    verbose: bool = True,
) -> Dict[str, object]:
    """
    End-to-end pipeline using:
    - one long DEG dataframe
    - one obs grouping column for expression summaries
    - optional layer selection for expression-based computations
    """
    markers_dict, gene_to_deg_group, unique_markers = select_unique_markers_from_deg_df(
        degs_df=degs_df,
        adata=adata,
        deg_group_col=deg_group_col,
        gene_col=gene_col,
        top_n=top_n,
        fdr_col=fdr_col,
        logfc_col=logfc_col,
        min_logfc=min_logfc,
        fdr_thresh=fdr_thresh,
        deg_group_order=deg_group_order,
        sort_by=sort_by,
        ascending=ascending,
        verbose=verbose,
    )

    stats_df = compute_marker_group_stats(
        adata=adata,
        markers_dict=markers_dict,
        obs_groupby=obs_groupby,
        marker_group_label=marker_group_label,
        layer=layer,
        positive_threshold=positive_threshold,
    )

    avg_expr = compute_group_average_expression(
        adata=adata,
        genes=unique_markers,
        obs_groupby=obs_groupby,
        layer=layer,
    )

    ordered_groups, corr = order_groups_by_correlation(avg_expr=avg_expr)
    ordered_genes = order_genes_by_dominant_group(
        avg_expr=avg_expr,
        ordered_groups=ordered_groups,
    )

    return {
        "markers_dict": markers_dict,
        "gene_to_deg_group": gene_to_deg_group,
        "unique_markers": unique_markers,
        "stats_df": stats_df,
        "avg_expr": avg_expr,
        "group_correlation": corr,
        "ordered_groups": ordered_groups,
        "ordered_genes": ordered_genes,
    }

In [ ]:
%load_ext rpy2.ipython
pd.DataFrame.iteritems = pd.DataFrame.items

# Try scanpy first

In [ ]:
MyeloidsSS = adataNiches[~adataNiches.obs["contrastCol"].str.contains("_Niche")].copy()
MyeloidsSS.X = MyeloidsSS.layers["counts"].copy()
rsc.get.anndata_to_GPU(MyeloidsSS)
rsc.pp.normalize_total(MyeloidsSS)
MyeloidsSS.layers["norm"] = MyeloidsSS.X.get()
rsc.pp.log1p(MyeloidsSS)
MyeloidsSS.layers["normLog"] = MyeloidsSS.X.get()
rsc.get.anndata_to_CPU(MyeloidsSS)
sc.tl.rank_genes_groups(MyeloidsSS, groupby="contrastCol", mehthod="wilcoxon", pts=True)
sc.tl.dendrogram(MyeloidsSS, groupby="contrastCol")
sc.pl.rank_genes_groups_dotplot(MyeloidsSS, n_genes=10, values_to_plot="logfoldchanges", min_logfoldchange=.5, standard_scale="var", cmap="RdBu_r")
sc.pl.rank_genes_groups_matrixplot(MyeloidsSS, n_genes=10, values_to_plot="logfoldchanges", min_logfoldchange=.5, standard_scale="var", cmap="RdBu_r", vmin=-3, vmax=3)


maxFDR = 0.01
min_pct = 0.05
w_niche = .5
scanpyDEGs = pd.DataFrame()
for group in MyeloidsSS.obs["contrastCol"].unique():
    localDEGs = sc.get.rank_genes_groups_df(MyeloidsSS, group)
    localDEGs = localDEGs[localDEGs["pvals_adj"] < maxFDR].copy()
    localDEGs["AnnotatedDomain"] = group
    rank = (
        -np.log10(np.clip(localDEGs["pvals_adj"], localDEGs.loc[localDEGs["pvals_adj"] != 0,"pvals_adj"].min(), None))
        * np.sign(localDEGs["logfoldchanges"])
        * localDEGs["pct_nz_group"]
    )
    localDEGs["rank"] = rank
    scanpyDEGs = pd.concat([scanpyDEGs,localDEGs ])

saveDir = f"DEGS_{nb_name}_scanpy"
os.makedirs(saveDir, exist_ok=True)

for AnnotatedDomain in scanpyDEGs["AnnotatedDomain"].unique():
    localDegs = scanpyDEGs[scanpyDEGs["AnnotatedDomain"] == AnnotatedDomain].copy()  
    localDegs.to_excel(os.path.join(saveDir, f"DEGS_{AnnotatedDomain}_maxFDR{maxFDR}.xlsx"), index=False)


out = run_marker_selection_and_ordering(
    adata=MyeloidsSS,
    degs_df=scanpyDEGs,
    deg_group_col="AnnotatedDomain",   # who owns candidate marker genes
    obs_groupby="contrastCol",  # where expression is summarized
    gene_col="names",
    top_n=10,
    fdr_col="pvals_adj",layer="normLog", 
    logfc_col="logfoldchanges",sort_by="rank", ascending=False,
    min_logfc=0.5,
    fdr_thresh=0.01,
)



sc.pl.dotplot(
    MyeloidsSS,
    out["ordered_genes"],
    "contrastCol",
    categories_order=out["ordered_groups"],
    layer="normLog",
    cmap="RdBu_r",standard_scale="var",
)

sc.pl.matrixplot(
    MyeloidsSS,
    out["ordered_genes"],
    "contrastCol",
    categories_order=out["ordered_groups"],
    layer="normLog",
    cmap="RdBu_r",standard_scale="var",
)


# Now Seurat method

In [ ]:
from transferUtils import *

export_anndata_minimal(
    adata=MyeloidsSS,
    out_dir=f"./SeuratReady_{Celltype}_{DSname}",
    base=f"{Celltype}_{DSname}",
    layer="counts",                 # ignored for data, used only for shape checking
    obsm_key="Banksy_PCA_lambda0.8",
    replace_counts_with_zeros=False, # <- zeros
)

In [ ]:
%%R -i DSname -i Celltype  -i homeDir  -o CtMarkers -i maxFDR -i min_pct
library(Seurat)
library(dplyr)

source(paste0(homeDir,"/utils/transferUtils.R"))


# If needed, pin Python with SciPy before calling:
# reticulate::use_python("/usr/bin/python", required = TRUE)

sce <- load_export_as_sce(
  out_dir = sprintf("SeuratReady_%s_%s", Celltype, DSname),
  base = sprintf("%s_%s", Celltype, DSname),
  reduced_name = "Banksy_PCA_lambda0.8",
  add_coords_to_reduced = TRUE,
  include_coords_in_coldata = TRUE,
  counts_transpose = TRUE,
  sep = "\t"
)

SeuratObject <- as.Seurat(sce, counts = "counts", data = "counts")
SeuratObject <- NormalizeData(SeuratObject)
Idents(SeuratObject) <- "contrastCol"
# markers <- FindAllMarkers(SeuratObject, only.pos = TRUE)

CtMarkers<-FindAllMarkers(SeuratObject,only.pos = T,min.pct = min_pct)
CtMarkers<-CtMarkers[CtMarkers$p_val_adj<maxFDR,]

In [ ]:
CtMarkers

In [ ]:
saveDir = f"DEGS_{nb_name}_Seurat"
os.makedirs(saveDir, exist_ok=True)


rank = (
    -np.log10(np.clip(CtMarkers["p_val_adj"], CtMarkers.loc[CtMarkers["p_val_adj"] != 0,"p_val_adj"].min(), None))
    * np.sign(CtMarkers["avg_log2FC"])
    * CtMarkers["pct.1"]
)
CtMarkers["rank"] = rank

for AnnotatedDomain in CtMarkers["cluster"].unique():
    localDegs = CtMarkers[CtMarkers["cluster"] == AnnotatedDomain].copy()  
    localDegs.to_excel(os.path.join(saveDir, f"DEGS_{AnnotatedDomain}_maxFDR{maxFDR}_minPosRate{min_pct}.xlsx"), index=False)

In [ ]:
MarkersDict ={}
topN = 10
for i in CtMarkers["cluster"].unique():
    MarkersDict[i] = CtMarkers[CtMarkers["cluster"] == i].sort_values("avg_log2FC").tail(topN)["gene"].tolist()

MyeloidsSS.X = MyeloidsSS.layers["counts"].copy()


In [ ]:
sc.pl.matrixplot(
    MyeloidsSS,
    MarkersDict,
    "contrastCol",
    dendrogram=True,
    # colorbar_title="mean z-score",
    # layer="scaled",
    cmap="RdBu_r",standard_scale="var",
)


sc.pl.dotplot(
    MyeloidsSS,
    MarkersDict,
    "contrastCol",
    dendrogram=True,
    # colorbar_title="mean z-score",
    # layer="scaled",
    cmap="RdBu_r",standard_scale="var",
)


out = run_marker_selection_and_ordering(
    adata=MyeloidsSS,
    degs_df=CtMarkers,
    deg_group_col="cluster",   # who owns candidate marker genes
    obs_groupby="contrastCol",  # where expression is summarized
    gene_col="gene",
    top_n=10,
    fdr_col="p_val_adj",
    logfc_col="avg_log2FC",layer="normLog", sort_by="rank", ascending=False,
    min_logfc=0.5,
    fdr_thresh=0.01,
)


sc.pl.matrixplot(
    MyeloidsSS,
    out["ordered_genes"],
    "contrastCol",
    categories_order=out["ordered_groups"],
    # colorbar_title="mean z-score",
    layer="normLog",
    cmap="RdBu_r",standard_scale="var",
)

# And metacells

# First check number of genes expressed per gorup

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

start, stop, step = 0, 1, 0.05
Rates = np.round(np.arange(start, stop + 1e-12, step), 2)

rows = []
for cl in adataNiches.obs["contrastCol"].unique():
    local = adataNiches[adataNiches.obs["contrastCol"] == cl]
    X = local.X
    n_cells = local.n_obs

    # number of positive cells per gene
    if sp.issparse(X):
        pos_counts = np.asarray((X > 0).sum(axis=0)).ravel()
    else:
        pos_counts = (X > 0).sum(axis=0).ravel()

    # inclusive thresholds: need >= ceil(n_cells * rate)
    thr = np.ceil(n_cells * Rates).astype(int)
    thr[Rates == 0.0] = 1  # define 0.00 as ">=1 cell positive"

    PosList = (pos_counts[None, :] >= thr[:, None]).sum(axis=1)
    rows.append(pd.Series(PosList, index=Rates, name=cl))

PosCellsDF = pd.DataFrame(rows)   # rows=clusters, cols=Rates
PosCellsDF

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math


import numpy as np
import pandas as pd
import scipy.sparse as sp

start, stop, step = 0, 1, 0.05
Rates = np.round(np.arange(start, stop + 1e-12, step), 2)

rows = []
for cl in adataNiches.obs["contrastCol"].unique():
    local = adataNiches[adataNiches.obs["contrastCol"] == cl]
    X = local.X
    n_cells = local.n_obs

    # number of positive cells per gene
    if sp.issparse(X):
        pos_counts = np.asarray((X > 0).sum(axis=0)).ravel()
    else:
        pos_counts = (X > 0).sum(axis=0).ravel()

    # inclusive thresholds: need >= ceil(n_cells * rate)
    thr = np.ceil(n_cells * Rates).astype(int)
    thr[Rates == 0.0] = 1  # define 0.00 as ">=1 cell positive"

    PosList = (pos_counts[None, :] >= thr[:, None]).sum(axis=1)
    rows.append(pd.Series(PosList, index=Rates, name=cl))

PosCellsDF = pd.DataFrame(rows)   # rows=clusters, cols=Rates



# Ensure posrate is numeric + sorted
Rates = np.array(PosCellsDF.columns, dtype=float)
PosCellsDF = PosCellsDF.loc[:, Rates[np.argsort(Rates)]]
Rates = np.array(PosCellsDF.columns, dtype=float)

clusters = list(PosCellsDF.index)
n = len(clusters)

# number of cells per cluster (needs adataNiches in scope)
cell_counts = (
    adataNiches.obs["contrastCol"]
    .value_counts()
    .reindex(clusters)
    .astype(int)
)

# layout
ncols = min(2, n)
nrows = math.ceil(n / ncols)

step = float(np.median(np.diff(Rates))) if len(Rates) > 1 else 0.05
bar_w = 0.9 * step

ymax = float(np.nanmax(PosCellsDF.to_numpy()))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(6 * ncols, 2.8 * nrows),
    sharex=True, sharey=True
)
axes = np.atleast_1d(axes).ravel()

for i, cl in enumerate(clusters):
    ax = axes[i]
    y = PosCellsDF.loc[cl].to_numpy(dtype=float)

    bars = ax.bar(Rates, y, width=bar_w, align="center")
    ax.set_title(f"{cl} (n={cell_counts.loc[cl]})")
    ax.set_ylim(0, ymax * 1.05)

    # annotate #positive genes (bar height) on top of each bar
    for rect, val in zip(bars, y):
        ax.text(
            rect.get_x() + rect.get_width() / 2,
            rect.get_height() + ymax * 0.01,
            f"{int(val)}",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=6,
            clip_on=True
        )

# hide unused axes
for j in range(n, len(axes)):
    axes[j].axis("off")

# ticks/labels
for ax in axes[:n]:
    ax.set_xticks(Rates)
    ax.set_xticklabels([f"{r:.2f}" for r in Rates], rotation=90, ha="center")

# axis labels
for r in range(nrows):
    axes[r * ncols].set_ylabel("# genes")

for ax in axes[(nrows - 1) * ncols : nrows * ncols]:
    if ax.has_data():
        ax.set_xlabel("posrate")

fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
from scipy import sparse

clusters = adataNiches.obs["contrastCol"]

# If cluster is categorical, this preserves a stable order
cluster_levels = clusters.cat.categories if hasattr(clusters.dtype, "categories") else clusters.unique()

n_vars = adataNiches.n_vars
keep_mask = np.zeros(n_vars, dtype=bool)

for cl in cluster_levels:
    idx = (clusters == cl).to_numpy()
    n = int(idx.sum())
    if n == 0:
        continue

    Xc = adataNiches.X[idx, :]  # slice once

    if sparse.issparse(Xc):
        pos_counts = np.asarray((Xc > 0).sum(axis=0)).ravel()
    else:
        pos_counts = (Xc > 0).sum(axis=0)

    keep_mask |= (pos_counts >= min_pct * n)

KeptGenes = adataNiches.var_names[keep_mask].tolist()
print(len(KeptGenes))

In [ ]:
metacellsAdata = adataNiches.copy()
metacellsAdata.X = metacellsAdata.layers["counts"]
sc.pp.normalize_total(metacellsAdata)

metacellsAdata
metacellsAdata = deterministic_aggregation_k3(
    _adata_group = metacellsAdata,
    group="Myeloids",
    cellStateObs = "contrastCol",
    PseudoReplicates_per_group = 10,pca="Banksy_PCA_lambda0.8",
    method="k3Metacells",countsLayer=None,
    n_pcs=15)


In [ ]:
def sanitize_strings(value):
    if isinstance(value, str):
        return (
            value.replace('_', '')
            .replace('-', '')
            .replace('@', '')
            .replace(',', '')
            .replace('/', '')
            .replace(':', '')
            .replace('(', '')
            .replace(')', '')
            .replace(';', '')
            .replace(' ', '')
        )
    return value


metacellsAdata.X = metacellsAdata.layers["k3Metacells_summedCounts"].copy()
#sc.pp.normalize_total(metacellsAdata, 1e6)
Counts = metacellsAdata.to_df().T.copy()
Counts = Counts.loc[KeptGenes]


MD = metacellsAdata.obs.copy()
MD["cluster"] = MD["contrastCol"].astype(str)

# 1) Apply to all values in MD (every column)
MD = MD.applymap(sanitize_strings)

# 2) Sanitize column names and index of MD
MD.columns = MD.columns.map(sanitize_strings)
MD.index   = MD.index.map(sanitize_strings)

# 3) Sanitize column names and index of Counts
Counts.columns = Counts.columns.map(sanitize_strings)



MD["cluster"].value_counts()


In [ ]:
%%R -i MD -i Counts -o results -i w_niche

library(edgeR)
library(stats)

  # <<< set niche weight here (e.g. 0.30, 0.33, 0.5)


# -----------------------------
# Parse cluster/domain/type
# -----------------------------
cluster <- factor(MD$cluster)

# IMPORTANT: your cluster strings are sanitized like:
# DomainCNSResidentMyeloids / DomainCNSResidentNiche
type   <- ifelse(grepl("Niche$", as.character(cluster)), "Niche", "Myeloids")
domain <- sub("(Myeloids|Niche)$", "", as.character(cluster))

group <- factor(paste0(type, ".", domain))   # e.g. "Myeloids.DomainCNSResident"

# -----------------------------
# edgeR object + design
# -----------------------------
y <- DGEList(Counts)
y$samples$cluster <- cluster
y$samples$type    <- type
y$samples$domain  <- domain
y$samples$group   <- group

design <- model.matrix(~ 0 + group)
colnames(design) <- sub("^group", "", colnames(design))
colnames(design) <- make.names(colnames(design))

# -----------------------------
# filter + normalize + fit
# -----------------------------
keep.genes <- filterByExpr(y, design)
y <- y[keep.genes, , keep.lib.sizes = FALSE]

y <- normLibSizes(y)
y <- estimateDisp(y, design, robust=TRUE)
fit <- glmQLFit(y, design, robust=TRUE)

# -----------------------------
# domains with BOTH Myeloids and Niche
# -----------------------------
domains_my <- unique(domain[type == "Myeloids"])
domains_ni <- unique(domain[type == "Niche"])
domains <- sort(intersect(domains_my, domains_ni))

if (length(domains) < 2) {
  stop("Need at least 2 domains with BOTH Myeloids and Niche.")
}

# -----------------------------
# Build fixed-weight contrasts
#   Myeloids.d - [ w*Niche.d + (1-w)*mean(Myeloids.others) ]
# -----------------------------

mean_myeloids_term <- function(ds) {
  # mean of other myeloids coefficients
  paste0("(", paste(paste0("Myeloids.", ds), collapse=" + "), ")/", length(ds))
}

exprs <- list()
for (d in domains) {
  others <- setdiff(domains, d)
  if (length(others) < 1) next

  exprs[[paste0(d, "_Myeloids")]] <-
    paste0(
      "Myeloids.", d,
      " - (",
      w_niche, "*Niche.", d,
      " + ",
      (1 - w_niche), "*", mean_myeloids_term(others),
      ")"
    )

  print(exprs[[paste0(d, "_Myeloids")]])
}

contr <- do.call(makeContrasts, c(exprs, list(levels = design)))
print(contr)

# -----------------------------
# Run tests + collect results
# -----------------------------
get_toptags_list <- function(fit, contr, n = Inf, sort.by = "logFC", ...) {
  out <- setNames(vector("list", ncol(contr)), colnames(contr))
  for (nm in colnames(contr)) {
    tt <- glmQLFTest(fit, contrast = contr[, nm])
    out[[nm]] <- topTags(tt, n = n, sort.by = sort.by, ...)$table
  }
  out
}

results <- get_toptags_list(fit, contr, n = 30000, sort.by = "logFC")


In [ ]:
resultsDict = dict(zip([str(i) for i in  list(results.names())], list(results.values())))
for k in resultsDict:
    resultsDict[k]["celltype"] = k
    resultsDict[k]["method"] = "DEA"
    resultsDict[k]["genes"] = resultsDict[k].index.tolist()
    resultsDict[k]["rank"] = np.sign(resultsDict[k]["logFC"])* -np.log10(resultsDict[k]["FDR"])
    resultsDict[k] = resultsDict[k].sort_values("rank", ascending=False)
    resultsDict[k] = resultsDict[k].reset_index()
    print(k)

agg_df, DEGsDictGO, fig = faceted_volcano(
    results_dict=resultsDict,
    fc="logFC", p="PValue", q="FDR", gene_col="genes",
    logfc_threshold=0.5, q_threshold=0.01,marker_size=8,width_scaleF=400,marker_line_width=0.4,
    q_trim=1, logfc_trim=np.inf,
    facet_col_wrap=4,  # change to 3/2 to make facets larger
    show=True
)


In [ ]:
combined = pd.concat(list(resultsDict.values()))
combined = combined[combined["FDR"] < maxFDR].copy()



saveDir = f"DEGS_{nb_name}_edgeR_nicheWeighting"
os.makedirs(saveDir, exist_ok=True)

for AnnotatedDomain in combined["celltype"].unique():
    localDegs = combined[combined["celltype"] == AnnotatedDomain].copy()  
    localDegs.to_excel(os.path.join(saveDir, f"DEGS_{AnnotatedDomain}_maxFDR{maxFDR}_nicheWight{w_niche}_minPosRate{min_pct}.xlsx"), index=False)


In [ ]:
out = run_marker_selection_and_ordering(
    adata=MyeloidsSS,
    degs_df=combined,
    deg_group_col="celltype",   # who owns candidate marker genes
    obs_groupby="contrastCol",  # where expression is summarized
    gene_col="genes",
    top_n=10,
    fdr_col="FDR",
    logfc_col="logFC",layer="normLog", 
    min_logfc=0.5,
    fdr_thresh=0.01,
)


sc.pl.dotplot(
    MyeloidsSS,
    out["ordered_genes"],
    "contrastCol",
    categories_order=out["ordered_groups"],
    # colorbar_title="mean z-score",
    layer="normLog",
    cmap="RdBu_r",standard_scale="var",
)

sc.pl.matrixplot(
    MyeloidsSS,
    out["ordered_genes"],
    "contrastCol",
    categories_order=out["ordered_groups"],
    # colorbar_title="mean z-score",
    layer="normLog",
    cmap="RdBu_r",standard_scale="var",
)

# Distance-based analysis

In [ ]:
adata.obs["AnnotatedDomain"].unique()

In [ ]:
adataDomains = adata[adata.obs["AnnotatedDomain"].isin(["MetastaticCore","InfiltratedStroma","CNS_Tumor_Rhim","ECM_Endo","PeritumoralStroma","PeritumoralStroma_Endothelium"])].copy()

adataDomains.obs["spacexr_refined"] = np.nan
adataDomains.obs["spacexr_refined"] = np.where(adataDomains.obs["spacexr"].str.contains("Tumor"), "Tumor",adataDomains.obs["spacexr"])

In [ ]:
adataDomains = adataDomains[~((adataDomains.obs["spacexr_refined"] == "Tumor") & (adataDomains.obs["AnnotatedDomain"] != "MetastaticCore"))]

adataDomains.obs["AnnotatedDomain_refined"] = "TumorProximal"

In [ ]:
assign_palette_topn_then_random(
    adataDomains,
    obs_key="spacexr_refined",
    palette_name="Set2",   # try "tab10", "Set3", "Accent", etc.
    random_seed=123
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
import squidpy as sq
import anndata as ad

def remove_ref_token(label, ref_token, sep=","):
    toks = [t.strip() for t in str(label).split(sep) if t.strip()]
    if ref_token in toks:
        toks = [t for t in toks if t != ref_token]
        return sep.join(toks) if len(toks) else ref_token
    return str(label)

def _ensure_palette(adata, obs_key):
    # Ensure categorical so categories align with *_colors
    if not pd.api.types.is_categorical_dtype(adata.obs[obs_key]):
        adata.obs[obs_key] = adata.obs[obs_key].astype("category")

    colors_key = f"{obs_key}_colors"
    if colors_key not in adata.uns:
        raise KeyError(f"Expected {colors_key} in adata.uns")

    cats = list(adata.obs[obs_key].cat.categories.astype(str))
    cols = list(adata.uns[colors_key])
    return dict(zip(cats, cols))


def prep_domain_selection_and_palette(
    adata,
    obs_key="spacexr_refined",
    domain_key="AnnotatedDomain",
    spatial_key="FULLRESspatial_microns",
    minIDsToKeep=200,
    light_uns_key=None,
    palette_uns_key=None,
    selected_uns_key=None,
    global_order_uns_key=None,
    domains_uns_key=None,
    make_panel1=True,
    figsize_per_domain=(30, 4.0),
):
    if light_uns_key is None:
        light_uns_key = f"{obs_key}_light_by_domain"
    if palette_uns_key is None:
        palette_uns_key = f"{obs_key}_palette"
    if selected_uns_key is None:
        selected_uns_key = f"{obs_key}_domain_selected"
    if global_order_uns_key is None:
        global_order_uns_key = f"{obs_key}_global_order_union"
    if domains_uns_key is None:
        domains_uns_key = f"{obs_key}_domains"

    palette = _ensure_palette(adata, obs_key)
    domains = sorted(adata.obs[domain_key].dropna().unique())
    domains_str = [str(d) for d in domains]

    # per-domain selected labels
    domain_selected = {}
    union_selected = set()

    for dom in domains_str:
        dom_obs = adata.obs.loc[adata.obs[domain_key].astype(str) == dom, obs_key].astype(str)
        vc = dom_obs.value_counts()
        sel = vc[vc > minIDsToKeep].index.tolist()
        domain_selected[dom] = sel
        union_selected.update(sel)

    # stable union order: by global frequency restricted to union
    global_vc = adata.obs[obs_key].astype(str).value_counts()
    global_order = [x for x in global_vc.index.tolist() if x in union_selected]

    # store light AnnData per domain (subset to selected labels within that domain)
    light_by_domain = {}
    for dom in domains_str:
        sel = domain_selected[dom]
        if len(sel) == 0:
            # empty light object
            ad0 = ad.AnnData(
                X=np.zeros((0, 0), dtype=np.float32),
                obs=pd.DataFrame(index=[]),
                var=pd.DataFrame(index=[]),
            )
            ad0.uns.update({"domain": dom, "obs_key": obs_key, "domain_key": domain_key, "spatial_key": spatial_key, "selected_ids": []})
            light_by_domain[dom] = ad0
            continue

        mask_dom = (adata.obs[domain_key].astype(str) == dom)
        mask_sel = adata.obs[obs_key].astype(str).isin(sel)
        keep_mask = mask_dom & mask_sel

        # build light AnnData (no var/no X)
        obs_sub = adata.obs.loc[keep_mask].copy()
        ad_light = ad.AnnData(
            X=np.zeros((obs_sub.shape[0], 0), dtype=np.float32),
            obs=obs_sub,
            var=pd.DataFrame(index=[]),
        )
        if spatial_key in adata.obsm:
            ad_light.obsm[spatial_key] = np.asarray(adata.obsm[spatial_key])[keep_mask.values, :]
        ad_light.uns.update({"domain": dom, "obs_key": obs_key, "domain_key": domain_key, "spatial_key": spatial_key, "selected_ids": sel})
        light_by_domain[dom] = ad_light

    # store in adata.uns for reuse
    adata.uns[palette_uns_key] = palette
    adata.uns[selected_uns_key] = domain_selected
    adata.uns[global_order_uns_key] = global_order
    adata.uns[domains_uns_key] = domains_str
    adata.uns[light_uns_key] = light_by_domain

    # ---- Panel 1 plotting (counts across union categories)
    fig = None
    if make_panel1:
        n_dom = len(domains_str)
        fig, axes = plt.subplots(
            n_dom, 1,
            figsize=(figsize_per_domain[0], figsize_per_domain[1] * n_dom),
            sharex=True,
            constrained_layout=True
        )
        if n_dom == 1:
            axes = [axes]

        for i, dom in enumerate(domains_str):
            ax = axes[i]
            dom_obs_all = adata.obs.loc[adata.obs[domain_key].astype(str) == dom, obs_key].astype(str)
            vc_dom = dom_obs_all.value_counts().reindex(global_order, fill_value=0)

            df = vc_dom.rename("count").reset_index().rename(columns={"index": obs_key})
            df[obs_key] = pd.Categorical(df[obs_key], categories=global_order, ordered=True)

            sns.barplot(
                data=df, x=obs_key, y="count",
                hue=obs_key, palette=palette,
                order=global_order, hue_order=global_order,
                dodge=False, ax=ax
            )

            # black outlines + labels
            for p in ax.patches:
                p.set_edgecolor("black")
                p.set_linewidth(0.8)
                h = p.get_height()
                if h > 0:
                    ax.annotate(
                        f"{int(h)}",
                        (p.get_x() + p.get_width()/2, h),
                        ha="center", va="bottom",
                        fontsize=7, rotation=90,
                        xytext=(0, 2), textcoords="offset points",
                        clip_on=True
                    )

            leg = ax.get_legend()
            if leg is not None:
                leg.remove()

            ax.set_title(dom)
            ax.set_xlabel(obs_key)
            ax.set_ylabel("n cells")
            ax.grid(False); ax.xaxis.grid(False); ax.yaxis.grid(False)

            if i != n_dom - 1:
                ax.tick_params(axis="x", labelbottom=False)
            else:
                ax.tick_params(axis="x", rotation=90)

    return palette, domain_selected, global_order, domains_str, light_by_domain, fig


def compute_domain_centrality_and_ref(
    adata,
    obs_key="spacexr_refined",
    spatial_key="FULLRESspatial_microns",
    light_uns_key=None,
    centrality_uns_key=None,

    # --- explicit squidpy neighbor params (defaults) ---
    coord_type="generic",
    radius=(10, 100),
    delaunay=True,
    n_neighs=None,

    # optional: pass any extra squidpy args directly
    neighbors_kwargs_extra=None,

    make_panel2=True,
    figsize_per_domain=(30, 3.2),
):
    if light_uns_key is None:
        light_uns_key = f"{obs_key}_light_by_domain"
    if centrality_uns_key is None:
        centrality_uns_key = f"{obs_key}_centrality_by_domain"

    palette = adata.uns.get(f"{obs_key}_palette", None)
    if palette is None:
        palette = _ensure_palette(adata, obs_key)
        adata.uns[f"{obs_key}_palette"] = palette

    light_by_domain = adata.uns.get(light_uns_key, None)
    if light_by_domain is None:
        raise KeyError(f"Missing adata.uns[{light_uns_key!r}]. Run prep_domain_selection_and_palette first.")

    metrics = ["degree_centrality", "average_clustering", "closeness_centrality"]
    centrality_by_domain = {}
    domains = list(light_by_domain.keys())

    # Panel 2 plotting (optional): rows=domains, cols=3 metrics
    fig = None
    axes = None
    if make_panel2:
        n_dom = len(domains)
        fig, axes = plt.subplots(
            n_dom, 3,
            figsize=(figsize_per_domain[0], figsize_per_domain[1] * n_dom),
            sharex=False, sharey=False,
            gridspec_kw={"wspace": 0.25, "hspace": 0.35},
            constrained_layout=True
        )
        if n_dom == 1:
            axes = np.array([axes])

    for i, dom in enumerate(domains):
        ad_light = light_by_domain[dom]
        sel = ad_light.uns.get("selected_ids", [])

        if ad_light.n_obs == 0 or len(sel) == 0 or spatial_key not in ad_light.obsm:
            centrality_by_domain[dom] = None
            if make_panel2:
                for j in range(3):
                    axes[i, j].set_axis_off()
            continue

        # ---- CRITICAL FIX: drop unused categories so squidpy doesn't iterate over empty clusters
        if pd.api.types.is_categorical_dtype(ad_light.obs[obs_key]):
            ad_light.obs[obs_key] = ad_light.obs[obs_key].cat.remove_unused_categories()
        else:
            ad_light.obs[obs_key] = ad_light.obs[obs_key].astype("category")

        # ---- neighbors (explicit args; no internal kwargs dict)
        if neighbors_kwargs_extra is None:
            neighbors_kwargs_extra = {}

        if n_neighs is None:
            sq.gr.spatial_neighbors(
                ad_light,
                spatial_key=spatial_key,
                coord_type=coord_type,
                radius=radius,
                delaunay=delaunay,
                **neighbors_kwargs_extra
            )
        else:
            sq.gr.spatial_neighbors(
                ad_light,
                spatial_key=spatial_key,
                coord_type=coord_type,
                radius=radius,
                delaunay=delaunay,
                n_neighs=n_neighs,
                **neighbors_kwargs_extra
            )

        # ---- centralities (computes ALL metrics)
        sq.gr.centrality_scores(ad_light, obs_key)

        # usually f"{obs_key}_centrality_scores" e.g. "spacexr_centrality_scores"
        cent = ad_light.uns.get(f"{obs_key}_centrality_scores", None)
        if cent is None:
            cent = ad_light.uns.get("spacexr_centrality_scores", None)

        if cent is None:
            centrality_by_domain[dom] = None
            if make_panel2:
                for j in range(3):
                    axes[i, j].set_axis_off()
            continue

        if not isinstance(cent, pd.DataFrame):
            cent = pd.DataFrame(cent)

        # standardize ordering / typing and restrict to selected IDs
        cent.index = cent.index.astype(str)
        keep = [x for x in cent.index.tolist() if x in set(map(str, sel))]
        cent = cent.loc[keep, metrics]

        centrality_by_domain[dom] = cent
        ad_light.uns["centrality_scores"] = cent  # convenience

        # ---- plotting (3 panels: one per metric)
        if make_panel2:
            for j, m in enumerate(metrics):
                ax = axes[i, j]

                df = cent[[m]].reset_index().rename(columns={"index": obs_key, m: "centrality"})
                df[obs_key] = df[obs_key].astype(str)
                df = df.sort_values("centrality", ascending=False)

                order = df[obs_key].tolist()
                df[obs_key] = pd.Categorical(df[obs_key], categories=order, ordered=True)

                sns.barplot(
                    data=df,
                    y=obs_key, x="centrality",
                    hue=obs_key, palette=palette,
                    order=order, hue_order=order,
                    dodge=False, ax=ax
                )
                for p in ax.patches:
                    p.set_edgecolor("black")
                    p.set_linewidth(0.8)

                leg = ax.get_legend()
                if leg is not None:
                    leg.remove()

                ax.set_title(f"{dom} — {m}")
                ax.set_xlabel(m)
                ax.set_ylabel("")
                ax.tick_params(axis="y", labelsize=8)
                ax.grid(False)
                ax.xaxis.grid(False)
                ax.yaxis.grid(False)

    adata.uns[centrality_uns_key] = centrality_by_domain
    adata.uns[light_uns_key] = light_by_domain  # write back updated objects
    return centrality_by_domain, light_by_domain, fig




def compute_distances_to_ref_and_store_light(
    adata,
    ref_by_domain,  # <-- MANUAL dict domain->ref_label
    obs_key="spacexr_refined",
    spatial_key="FULLRESspatial_microns",
    light_uns_key=None,
    dist_uns_key=None,
    dist_quantile_trim=99,
    max_distance=None,  # <-- NEW (mutually exclusive with dist_quantile_trim)
    make_panel3=True,
    figsize_per_domain=(18, 3.5),
):
    # ---- validate mutually exclusive thresholds
    if (dist_quantile_trim is not None) and (max_distance is not None):
        raise ValueError("Provide only one of dist_quantile_trim or max_distance (they are mutually exclusive).")

    if max_distance is not None:
        max_distance = np.float32(max_distance)
        if not np.isfinite(max_distance) or max_distance <= 0:
            raise ValueError("max_distance must be a finite positive number.")

    if light_uns_key is None:
        light_uns_key = f"{obs_key}_light_by_domain"
    if dist_uns_key is None:
        dist_uns_key = f"{obs_key}_dist_to_ref_by_domain"

    palette = adata.uns.get(f"{obs_key}_palette", None)
    if palette is None:
        palette = _ensure_palette(adata, obs_key)
        adata.uns[f"{obs_key}_palette"] = palette

    light_by_domain = adata.uns.get(light_uns_key, None)
    if light_by_domain is None:
        raise KeyError(f"Missing adata.uns[{light_uns_key!r}]. Run prep_domain_selection_and_palette first.")

    dist_by_domain = {}
    domains = list(light_by_domain.keys())

    fig = None
    axes = None
    if make_panel3:
        n_dom = len(domains)
        fig, axes = plt.subplots(
            n_dom, 1,
            figsize=(figsize_per_domain[0], figsize_per_domain[1] * n_dom),
            sharex=True, sharey=False,
            constrained_layout=True
        )
        if n_dom == 1:
            axes = [axes]

    for i, dom in enumerate(domains):
        ad_light_in = light_by_domain[dom]
        sel = ad_light_in.uns.get("selected_ids", [])
        ref_label = ref_by_domain.get(dom, None)

        ax = axes[i] if make_panel3 else None

        if ad_light_in.n_obs == 0 or len(sel) == 0 or spatial_key not in ad_light_in.obsm or ref_label is None:
            dist_by_domain[dom] = ad.AnnData(
                X=np.zeros((0, 0), dtype=np.float32),
                obs=pd.DataFrame(index=[]),
                var=pd.DataFrame(index=[])
            )
            dist_by_domain[dom].uns.update({
                "domain": dom, "ref_label": ref_label, "spatial_key": spatial_key, "obs_key": obs_key,
                "dist_quantile_trim": dist_quantile_trim, "max_distance": max_distance
            })
            if make_panel3:
                ax.set_axis_off()
            continue

        coords = np.asarray(ad_light_in.obsm[spatial_key])
        raw_labels = ad_light_in.obs[obs_key].astype(str).values
        cell_ids = ad_light_in.obs_names.values

        ref_label = str(ref_label)
        ref_mask = (raw_labels == ref_label)
        if ref_mask.sum() == 0:
            dist_by_domain[dom] = ad.AnnData(
                X=np.zeros((0, 0), dtype=np.float32),
                obs=pd.DataFrame(index=[]),
                var=pd.DataFrame(index=[])
            )
            dist_by_domain[dom].uns.update({
                "domain": dom, "ref_label": ref_label, "spatial_key": spatial_key, "obs_key": obs_key,
                "dist_quantile_trim": dist_quantile_trim, "max_distance": max_distance
            })
            if make_panel3:
                ax.set_axis_off()
            continue

        tree = cKDTree(coords[ref_mask])

        # allowed parsed IDs derive strictly from selected IDs in THIS domain
        parsed_allowed = {remove_ref_token(x, ref_label) for x in sel if str(x) != ref_label}
        parsed_allowed.discard(ref_label)

        nonref_mask = ~ref_mask
        if nonref_mask.sum() == 0:
            dist_by_domain[dom] = ad.AnnData(
                X=np.zeros((0, 0), dtype=np.float32),
                obs=pd.DataFrame(index=[]),
                var=pd.DataFrame(index=[])
            )
            dist_by_domain[dom].uns.update({
                "domain": dom, "ref_label": ref_label, "spatial_key": spatial_key, "obs_key": obs_key,
                "dist_quantile_trim": dist_quantile_trim, "max_distance": max_distance
            })
            if make_panel3:
                ax.set_axis_off()
            continue

        dists, _ = tree.query(coords[nonref_mask], k=1)
        parsed_ids = np.array([remove_ref_token(lab, ref_label) for lab in raw_labels[nonref_mask]], dtype=object)

        dist_df = pd.DataFrame(
            {
                "parsed_id": parsed_ids,
                "dist_to_ref": dists.astype(np.float32),
                "raw_id": raw_labels[nonref_mask],
                "ref_id": ref_label,
                "domain": dom,
            },
            index=cell_ids[nonref_mask]
        )

        # keep only parsed ids you selected (same as before)
        dist_df = dist_df[dist_df["parsed_id"].isin(parsed_allowed)]

        # ---- NEW: max_distance trimming (global within domain, after parsed_id filter)
        if max_distance is not None and dist_df.shape[0] > 0:
            dist_df = dist_df.loc[dist_df["dist_to_ref"] <= max_distance]

        # ---- existing: quantile trim within each parsed_id (domain-wise)
        if dist_quantile_trim is not None and dist_df.shape[0] > 0:
            q = float(dist_quantile_trim) / 100.0

            def _trim(g):
                if g.shape[0] < 2:
                    return g
                thr = g["dist_to_ref"].quantile(q)
                return g[g["dist_to_ref"] <= thr]

            dist_df = dist_df.groupby("parsed_id", group_keys=False, sort=False).apply(_trim)

        if dist_df.shape[0] == 0:
            dist_by_domain[dom] = ad.AnnData(
                X=np.zeros((0, 0), dtype=np.float32),
                obs=pd.DataFrame(index=[]),
                var=pd.DataFrame(index=[])
            )
            dist_by_domain[dom].uns.update({
                "domain": dom, "ref_label": ref_label, "spatial_key": spatial_key, "obs_key": obs_key,
                "dist_quantile_trim": dist_quantile_trim, "max_distance": max_distance
            })
            if make_panel3:
                ax.set_axis_off()
            continue

        # store light output
        ad_out = ad.AnnData(
            X=np.zeros((dist_df.shape[0], 0), dtype=np.float32),
            obs=dist_df.copy(),
            var=pd.DataFrame(index=[])
        )
        ad_out.obsm["dist_to_ref"] = dist_df["dist_to_ref"].to_numpy(dtype=np.float32)[:, None]
        ad_out.uns.update({
            "domain": dom,
            "ref_label": ref_label,
            "spatial_key": spatial_key,
            "obs_key": obs_key,
            "dist_quantile_trim": dist_quantile_trim,
            "max_distance": max_distance,
        })
        dist_by_domain[dom] = ad_out

        # panel 3 plot
        if make_panel3:
            med_order = (
                dist_df.groupby("parsed_id")["dist_to_ref"]
                .median().sort_values(ascending=True).index.tolist()
            )
            plot_df = dist_df.copy()
            plot_df["parsed_id"] = pd.Categorical(plot_df["parsed_id"], categories=med_order, ordered=True)

            sns.boxplot(
                data=plot_df,
                y="parsed_id", x="dist_to_ref",
                order=med_order,
                palette=palette,
                ax=ax
            )
            # black outlines
            for artist in ax.artists:
                artist.set_edgecolor("black")
                artist.set_linewidth(0.8)
            for line in ax.lines:
                line.set_color("black")
                line.set_linewidth(0.8)

            ax.set_title(f"{dom} — dist to ref: {ref_label}")
            ax.set_xlabel(f"nearest distance ({spatial_key})")
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelsize=8)
            ax.grid(False); ax.xaxis.grid(False); ax.yaxis.grid(False)

            if i != len(domains) - 1:
                ax.tick_params(axis="x", labelbottom=False)

    adata.uns[dist_uns_key] = dist_by_domain
    return dist_by_domain, fig

In [ ]:

# Step 1: prep + Panel 1 + light subsets
palette, domain_selected, global_order, domains, light_by_domain, fig1 = prep_domain_selection_and_palette(
    adataDomains,
    obs_key="spacexr_refined",
    domain_key="AnnotatedDomain_refined",
    spatial_key="FULLRESspatial_microns",figsize_per_domain=(10,3),
    minIDsToKeep=200,   # renamed
    make_panel1=True,
)


# Step 2: centralities + Panel 2 (all 3 metrics)
centrality_by_domain, light_by_domain, fig2 = compute_domain_centrality_and_ref(
    adataDomains,
    obs_key="spacexr_refined",
    spatial_key="FULLRESspatial_microns",
    make_panel2=True,
)

plt.show()

In [ ]:
# ---- you intervene here and provide manual refs:
ref_by_domain = {
    "TumorProximal": "Tumor",
}

# Step 3: distances + Panel 3 using manual refs
dist_by_domain, fig3 = compute_distances_to_ref_and_store_light(
    adataDomains,
    ref_by_domain=ref_by_domain,
    obs_key="spacexr_refined",
    spatial_key="FULLRESspatial_microns",figsize_per_domain=(10,10),
    max_distance=300,
    make_panel3=True,dist_quantile_trim=None
)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

def _make_dist_bins(dist_series, n_bins=10, round_to=1, method="cut"):
    """
    Return a string Series of bin centers computed from dist_series only.
    method: "cut" (equal width) or "qcut" (equal frequency).
    """
    x = pd.to_numeric(dist_series, errors="coerce")

    out = pd.Series(index=dist_series.index, data=np.nan, dtype="object")

    if x.notna().sum() == 0:
        return out, None  # no meta

    if x.nunique(dropna=True) < 2:
        center = float(np.round(x.dropna().iloc[0], round_to))
        out.loc[:] = str(center)
        meta = {"edges": None, "centers": [center], "method": "constant"}
        return out, meta

    if method == "qcut":
        # quantile-based bins
        b = pd.qcut(x, q=n_bins, duplicates="drop")
        # bin centers from bin midpoints
        centers = np.array([(iv.left + iv.right) / 2 for iv in b.cat.categories], dtype=float)
        # map each interval to its center
        center_map = {iv: c for iv, c in zip(b.cat.categories, centers)}
        out = b.map(center_map)
        out = np.round(out.astype(float), round_to).astype(str)
        meta = {"edges": None, "centers": centers.tolist(), "method": "qcut"}
        return out, meta

    elif method == "cut":
        # equal-width bins
        _, edges = pd.cut(x, bins=n_bins, retbins=True, include_lowest=True)
        centers = (edges[:-1] + edges[1:]) / 2
        binned_center = pd.cut(
            x,
            bins=edges,
            labels=centers,
            include_lowest=True,
        ).astype(float)
        out = np.round(binned_center, round_to).astype(str)
        meta = {"edges": edges.tolist(), "centers": centers.tolist(), "method": "cut"}
        return out, meta

    else:
        raise ValueError("method must be 'cut' or 'qcut'")


def add_distance_bins_labelwise_inplace(
    *,
    adata,
    dist_by_domain,
    obs_keys,
    dist_col="dist_to_ref",
    n_bins=10,
    round_bin_center=1,
    bin_method="cut",      # "cut" or "qcut"
    transform=None,        # None or "log1p"
    bin_col_prefix="dist_bin_center__",
    store_meta=True,
):
    """
    Updates dist_by_domain[dom].obs by adding one column per obs_key:
        f"{bin_col_prefix}{obs_key}"
    Bin assignment is computed label-wise:
        within each domain, within each obs_key, within each label.
    Also ensures dist_ad.obs has obs_key columns (pulled from adata.obs by index).
    """

    if isinstance(obs_keys, str):
        obs_keys = [obs_keys]
    obs_keys = list(obs_keys)

    for dom, dist_ad in dist_by_domain.items():
        if dist_ad is None or dist_ad.n_obs == 0:
            continue

        # Ensure obs_keys are present in dist_ad.obs (pull from full adata by barcode)
        idx = dist_ad.obs_names
        for ok in obs_keys:
            if ok not in dist_ad.obs.columns:
                if ok not in adata.obs.columns:
                    raise KeyError(f"obs_key {ok!r} not in adata.obs")
                dist_ad.obs[ok] = adata.obs.loc[idx, ok].astype(str).values

        # Optional metadata container
        if store_meta:
            dist_ad.uns.setdefault("distance_binning", {})
            dist_ad.uns["distance_binning"].setdefault(dom, {})
            dom_meta = dist_ad.uns["distance_binning"][dom]
        else:
            dom_meta = None

        # Base distance vector (optionally transformed)
        base_x = pd.to_numeric(dist_ad.obs[dist_col], errors="coerce")
        if transform == "log1p":
            base_x = np.log1p(base_x.astype(float))

        for ok in obs_keys:
            bin_col = f"{bin_col_prefix}{ok}"
            labels = dist_ad.obs[ok].astype(str)

            # allocate result
            out_bins = pd.Series(index=dist_ad.obs_names, data=np.nan, dtype="object")

            if store_meta:
                dom_meta.setdefault(ok, {})
                ok_meta = dom_meta[ok]
                ok_meta["transform"] = transform
                ok_meta["bin_method"] = bin_method
                ok_meta["n_bins"] = int(n_bins)
                ok_meta["round_bin_center"] = int(round_bin_center)
                ok_meta.setdefault("per_label", {})
            else:
                ok_meta = None

            # label-wise binning
            for lab in labels.unique():
                sel = (labels.values == lab)
                if sel.sum() == 0:
                    continue

                lab_bins, meta = _make_dist_bins(
                    base_x[sel],
                    n_bins=n_bins,
                    round_to=round_bin_center,
                    method=bin_method,
                )
                out_bins.loc[labels.index[sel]] = lab_bins.values

                if store_meta:
                    ok_meta["per_label"][lab] = meta

            dist_ad.obs[bin_col] = out_bins.values

    return dist_by_domain

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


def plot_cells_per_bin_by_label_panels(
    dist_ad,  # dist_by_domain["TumorProximal"] (AnnData)
    *,
    label_col="parsed_id",
    bin_col="dist_bin_center__spacexr_refined",
    ncols=1,
    figsize_per_panel=(14, 2.2),
    rotate="auto",
    tick_fontsize=8,
    min_total_cells=1,
    max_labels=None,
    sort_labels_by="total_cells",   # "total_cells" | "name"
    sort_bins_numeric=True,
    annotate_counts=False,
):
    obs = dist_ad.obs.copy()
    for c in (label_col, bin_col):
        if c not in obs.columns:
            raise KeyError(f"Missing {c!r} in dist_ad.obs")

    # counts per (label, bin)
    counts = (
        obs.groupby([label_col, bin_col], dropna=False)
           .size()
           .reset_index(name="n_cells")
           .dropna(subset=[label_col, bin_col])
    )

    # filter labels
    totals = counts.groupby(label_col)["n_cells"].sum()
    keep_labels = totals[totals >= min_total_cells].index.tolist()

    if sort_labels_by == "total_cells":
        keep_labels = totals.loc[keep_labels].sort_values(ascending=False).index.tolist()
    elif sort_labels_by == "name":
        keep_labels = sorted(keep_labels)
    else:
        raise ValueError("sort_labels_by must be 'total_cells' or 'name'")

    if max_labels is not None:
        keep_labels = keep_labels[:max_labels]

    counts = counts[counts[label_col].isin(keep_labels)].copy()

    # global bin order (shared across panels)
    bin_vals = counts[bin_col].astype(str)
    if sort_bins_numeric:
        bin_num = pd.to_numeric(bin_vals, errors="coerce")
        if bin_num.notna().all():
            ordered_bins = [str(b) for b in pd.Series(bin_num.unique()).sort_values().tolist()]
        else:
            ordered_bins = sorted(bin_vals.unique())
    else:
        ordered_bins = sorted(bin_vals.unique())

    counts[bin_col] = counts[bin_col].astype(str)
    counts[bin_col] = pd.Categorical(counts[bin_col], categories=ordered_bins, ordered=True)

    # layout
    n_panels = len(keep_labels)
    nrows = int(np.ceil(n_panels / ncols))
    fig_w = figsize_per_panel[0] * ncols
    fig_h = figsize_per_panel[1] * nrows

    with sns.axes_style("whitegrid"), sns.plotting_context("talk", font_scale=0.85):
        fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), constrained_layout=True)
        axes = np.array(axes).reshape(-1)

        for ax, lab in zip(axes, keep_labels):
            df = counts[counts[label_col] == lab].copy()

            sns.barplot(
                data=df,
                x=bin_col, y="n_cells",
                order=ordered_bins,
                ax=ax,
                edgecolor="black",
                linewidth=0.8,
            )

            total = int(df["n_cells"].sum())
            ax.set_title(f"{lab}  (n={total})", pad=6)
            ax.set_xlabel("")
            ax.set_ylabel("# cells")
            sns.despine(ax=ax, top=True, right=True)

            ax.set_axisbelow(True)
            ax.grid(axis="y", linestyle="-", linewidth=0.6, alpha=0.35)
            ax.grid(axis="x", visible=False)

            # ✅ show ALL bins (no downsampling)
            ax.set_xticks(np.arange(len(ordered_bins)))
            ax.set_xticklabels(ordered_bins, fontsize=tick_fontsize)

            # rotation logic
            if rotate == "auto":
                angle = 0 if len(ordered_bins) <= 12 else 90
            else:
                angle = rotate
            if angle is not None:
                ax.tick_params(axis="x", rotation=angle)

            # optional bar annotations
            if annotate_counts:
                for p in ax.patches:
                    h = p.get_height()
                    if h > 0:
                        ax.annotate(
                            f"{int(h)}",
                            (p.get_x() + p.get_width()/2, h),
                            ha="center", va="bottom",
                            fontsize=7,
                            xytext=(0, 2), textcoords="offset points",
                            clip_on=True,
                        )

        # turn off unused axes
        for ax in axes[n_panels:]:
            ax.set_axis_off()

    return fig, counts


# Example:
# fig, counts_df = plot_cells_per_bin_by_label_panels(
#     dist_by_domain["TumorProximal"],
#     label_col="parsed_id",
#     bin_col="dist_bin_center__spacexr_refined",
#     ncols=1,
#     figsize_per_panel=(14, 2.1),
#     tick_fontsize=7,
#     rotate=90,              # if you know bins are many, force rotation
#     min_total_cells=50,
# )
# plt.show()

In [ ]:
dist_by_domain = add_distance_bins_labelwise_inplace(
    adata=adataDomains,
    dist_by_domain=dist_by_domain,
    obs_keys=["spacexr_refined"],   # or multiple keys
    n_bins=10,
    round_bin_center=1,
    bin_method="cut",              # or "qcut"
    transform=None,                # or "log1p"
)

fig, counts_df = plot_cells_per_bin_by_label_panels(
    dist_by_domain["TumorProximal"],
    label_col="parsed_id",
    bin_col="dist_bin_center__spacexr_refined",
    ncols=1,
    figsize_per_panel=(14, 2.1),
)

In [ ]:
celltype = "Myeloids"
celltypeObs = dist_by_domain["TumorProximal"][dist_by_domain["TumorProximal"].obs["parsed_id"] == celltype].obs.copy()
celltypeAdata = adataDomains[celltypeObs.index].copy()
celltypeAdata.X = celltypeAdata.layers["counts"].copy()
celltypeAdata.obs["dist_bin_center__spacexr_refined"] = celltypeObs["dist_bin_center__spacexr_refined"] 
sc.pp.normalize_total(celltypeAdata)
celltypeAdata.obs["dist_bin_center__spacexr_refined"] = celltypeAdata.obs["dist_bin_center__spacexr_refined"].astype(np.float32)

In [ ]:
assign_palette_topn_then_random(
    celltypeAdata,
    obs_key="dist_bin_center__spacexr_refined",palette_is_continuous=True,
    palette_name="rainbow",   # try "tab10", "Set3", "Accent", etc.
    random_seed=123
)



plot_spatial_obs(
    celltypeAdata, obs="dist_bin_center__spacexr_refined", width=7, dpi=150,dotscale=3 ,img_key="hires",marker='o',
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", enforce_pixel_spot_size=False, img_alpha=.4,  legend_kwargs={"fontsize":10})

In [ ]:
celltypeMetacellsAdata = deterministic_aggregation_k3(
    _adata_group = celltypeAdata,
    group=celltype,
    cellStateObs = "dist_bin_center__spacexr_refined",
    PseudoReplicates_per_group = 10,pca="FULLRESspatial_microns",
    method="k3Metacells",countsLayer=None,verbose=False,
    n_pcs=2)

In [ ]:
celltypeMetacellsAdata.layers["counts"] = celltypeMetacellsAdata.X.copy()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


def histplot_all_observed_ticks(
    values,
    *,
    ax=None,                 # <-- NEW
    figsize=(10, 4),
    discrete=True,
    max_xticks=35,
    rotate="auto",
    tick_fontsize=9,
    title=None,
    xlabel=None,
    ylabel="Count",
    show_kde=False,
    color=None,
):
    x = pd.Series(values).dropna()

    if discrete:
        if np.all(np.isclose(x.astype(float), np.round(x.astype(float)))):
            x = np.round(x.astype(float)).astype(int)

    uniq = np.sort(pd.unique(x))

    # Create fig/ax only if not provided
    created_fig = False
    if ax is None:
        with sns.axes_style("whitegrid"), sns.plotting_context("talk", font_scale=0.9):
            fig, ax = plt.subplots(figsize=figsize)
        created_fig = True
    else:
        fig = ax.figure

    sns.histplot(
        x=x,
        discrete=discrete,
        kde=show_kde,
        ax=ax,
        color=color,
        edgecolor="black",
        linewidth=0.8,
        alpha=0.9,
    )

    sns.despine(ax=ax, top=True, right=True)

    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="-", linewidth=0.6, alpha=0.35)
    ax.grid(axis="x", visible=False)

    if xlabel is None:
        xlabel = getattr(values, "name", "Value")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title, pad=8)

    if len(uniq) <= max_xticks:
        ticks = uniq
    else:
        step = int(np.ceil(len(uniq) / max_xticks))
        ticks = uniq[::step]
    ax.set_xticks(ticks)

    if rotate == "auto":
        angle = 0 if len(ticks) <= 12 else 90
    else:
        angle = rotate
    if angle is not None:
        ax.tick_params(axis="x", rotation=angle)
    ax.tick_params(axis="x", labelsize=tick_fontsize)

    if created_fig:
        fig.tight_layout()

    return fig, ax


fig, ax = histplot_all_observed_ticks(
    celltypeMetacellsAdata.obs["nCells"],
    figsize=(14, 4),
    max_xticks=30,
    title="Metacell sizes (nCells)",
    xlabel="nCells",
)

In [ ]:
sc.pp.normalize_total(celltypeMetacellsAdata)

sc.pp.log1p(celltypeMetacellsAdata)

sc.tl.pca(celltypeMetacellsAdata)

celltypeMetacellsAdata.obs["dist_bin_center__spacexr_refined"] = celltypeMetacellsAdata.obs["dist_bin_center__spacexr_refined"].astype(np.float32)
assign_palette_topn_then_random(
    celltypeMetacellsAdata,
    obs_key="dist_bin_center__spacexr_refined",palette_is_continuous=True,
    palette_name="rainbow",   # try "tab10", "Set3", "Accent", etc.
    random_seed=123
)



from PlotPCA_components import *

plotPCA_components(celltypeMetacellsAdata, color="dist_bin_center__spacexr_refined", dotsize=200)

In [ ]:
plotPCA_components(celltypeMetacellsAdata, color="MBP", dotsize=200)

# GLM fitting

In [ ]:
from transferUtils import *

export_anndata_minimal(
    adata=celltypeMetacellsAdata,
    out_dir=f"./Trajectory_Ready_{Celltype}_{DS}",
    base=f"{Celltype}",
    layer="counts",                 # ignored for data, used only for shape checking
    obsm_key="X_pca",
    replace_counts_with_zeros=False, # <- zeros
)

In [ ]:
%%R -i homeDir -i Celltype -i DS -o tab  -o shrinkedSIgnificance -o logCPM_obs -o logCPM_fit_bins  -o logCPM_fitDF -o mono_table

# -----------------------------
# Load data (SCE) from your export
# -----------------------------
library(edgeR)
library(SingleCellExperiment)
library(Matrix)
library(splines)

source(paste0(homeDir, "/utils/transferUtils.R"))
out_dir <- file.path(sprintf("./Trajectory_Ready_%s_%s", Celltype,DS))

sce <- load_export_as_sce(
  out_dir = out_dir,
  base = Celltype,
  reduced_name = "X_pca",
  add_coords_to_reduced = TRUE,
  include_coords_in_coldata = TRUE,
  counts_transpose = TRUE,
  sep = "\t"
)

# -----------------------------
# Counts + covariate
# -----------------------------
cts <- assay(sce, "counts")  # genes x cells (can be dgCMatrix)

cd <- as.data.frame(colData(sce))
dist <- as.numeric(cd$dist_bin_center__spacexr_refined)

# Ensure alignment: columns of counts == rows of colData
stopifnot(identical(colnames(cts), rownames(colData(sce))))

# Drop NA distances (recommended)
keep_cells <- !is.na(dist)
if (!all(keep_cells)) {
  print("Filtering cells")
  cts  <- cts[, keep_cells, drop = FALSE]
  dist <- dist[keep_cells]
}

# -----------------------------
# edgeR GLM with spline(dist)
# -----------------------------
y <- DGEList(counts = cts, group=dist)
y$samples$dist_bin_center__spacexr_refined <- dist

# (Optional) keep only expressed genes for speed/stability
# keep_genes <- filterByExpr(y)
# print(table(keep_genes))
# y <- y[keep_genes, , keep.lib.sizes = FALSE]

y <- normLibSizes(y)

# Spline basis; df must match downstream endpoint contrast code
X <- ns(dist, df = 3)
design <- model.matrix(~ X)

y <- estimateDisp(y, design, robust = TRUE)

fit <- glmQLFit(y, design, robust = TRUE)
qlf <- glmQLFTest(fit, coef = 2:4)



tab <- as.data.frame(topTags(qlf, n = 30000))


# NOTE: decideTests() is really meant for multiple contrasts (glmTreat).
# Keeping it because you export it, but treat it as a coarse call.
shrinkedSIgnificance <- as.data.frame(decideTests(qlf), lfc = 1)

# Observed logCPM per cell
logCPM_obs <- edgeR::cpm(y, log = TRUE, prior.count = fit$prior.count)

# Fitted logCPM per cell (same dims as y$counts)
logCPM_fit <- edgeR::cpm(fit, log = TRUE)

logCPM_fitDF <- as.data.frame(logCPM_fit)
colnames(logCPM_fitDF) <- colnames(y)  # cell IDs

# -----------------------------
# Collapse fitted values to unique distance "bins" (unique dist values)
# -----------------------------
dist_key <- sprintf("%.3f", dist)         # stable string key
keep_bins <- !duplicated(dist_key)

logCPM_fit_bins <- logCPM_fit[, keep_bins, drop = FALSE]
colnames(logCPM_fit_bins) <- dist_key[keep_bins]
logCPM_fit_bins <- as.data.frame(logCPM_fit_bins)

# -----------------------------
# Monotonic genes + endpoint statistical test
#   - monotonicity uses fitted bin profiles (logCPM units)
#   - endpoint test uses the same spline model (QLF contrast end vs start)
# -----------------------------
# order bins numerically
bin_num <- as.numeric(colnames(logCPM_fit_bins))
ord <- order(bin_num)
logCPM_fit_bins_ord <- as.matrix(logCPM_fit_bins[, ord, drop = FALSE])

# monotonicity settings
tol <- 0.02       # allow tiny wiggles in logCPM
min_delta <- 0.25 # require abs change (first->last bin) in logCPM

is_mono_increasing <- function(v, tol=0) all(diff(v) >= -tol)
is_mono_decreasing <- function(v, tol=0) all(diff(v) <=  tol)

mono_inc <- apply(logCPM_fit_bins_ord, 1, is_mono_increasing, tol = tol)
mono_dec <- apply(logCPM_fit_bins_ord, 1, is_mono_decreasing, tol = tol)
mono_any <- mono_inc | mono_dec

delta <- logCPM_fit_bins_ord[, ncol(logCPM_fit_bins_ord)] - logCPM_fit_bins_ord[, 1]
mono_any <- mono_any & (abs(delta) >= min_delta)

# Endpoint contrast: fitted mean at max(dist) - min(dist) using same ns(df=3)
dist_start <- min(dist, na.rm = TRUE)
dist_end   <- max(dist, na.rm = TRUE)

X_basis <- ns(dist, df = 3)
X_new <- predict(X_basis, newx = c(dist_start, dist_end))  # 2 x df

row_start <- c(1, X_new[1, ])
row_end   <- c(1, X_new[2, ])
contr_end_vs_start <- row_end - row_start

test_end_start <- glmQLFTest(fit, contrast = contr_end_vs_start)
tt_end_start <- topTags(test_end_start, n = Inf)$table
tt_end_start$FDR_end_start <- p.adjust(tt_end_start$PValue, method = "BH")

# build mono_table
mono_table <- data.frame(
  gene = rownames(logCPM_fit_bins_ord),
  mono_increasing = mono_inc,
  mono_decreasing = mono_dec,
  delta_first_last = delta,
  stringsAsFactors = FALSE
)

mono_table <- mono_table[mono_any, , drop = FALSE]

# merge endpoint stats safely by gene name
mono_table$logFC_end_start  <- tt_end_start[mono_table$gene, "logFC"]
mono_table$PValue_end_start <- tt_end_start[mono_table$gene, "PValue"]
mono_table$FDR_end_start    <- tt_end_start[mono_table$gene, "FDR_end_start"]

# filter by endpoint FDR and direction consistency
alpha <- 0.05
mono_table <- mono_table[!is.na(mono_table$FDR_end_start) & mono_table$FDR_end_start < alpha, , drop = FALSE]

mono_table <- mono_table[
  (mono_table$mono_increasing & mono_table$logFC_end_start > 0) |
  (mono_table$mono_decreasing & mono_table$logFC_end_start < 0),
  , drop = FALSE
]

# sort by strongest endpoint effect (edgeR's logFC scale)
mono_table <- mono_table[order(-abs(mono_table$logFC_end_start)), , drop = FALSE]

cat("Monotone genes (after endpoint FDR):", nrow(mono_table), "\n")
cat("  increasing:", sum(mono_table$mono_increasing), "\n")
cat("  decreasing:", sum(mono_table$mono_decreasing), "\n")

In [ ]:
mono_table

In [ ]:
%%R

# 1) turn significance into a named vector
sig <- shrinkedSIgnificance[, 1]
names(sig) <- rownames(shrinkedSIgnificance)

# 2) align by gene names
common <- intersect(rownames(logCPM_fit_bins), names(sig))

# optional sanity print
cat("genes in logCPM_fit_bins:", nrow(logCPM_fit_bins), "\n")
cat("genes in shrinkedSIgnificance:", length(sig), "\n")
cat("common genes:", length(common), "\n")

# 3) reorder both to the same gene order
mat_aligned <- logCPM_fit_bins[common, , drop = FALSE]
sig_aligned <- sig[common]

# 4) filter rows (genes) where significant == 1
logCPM_fit_bins_Filt <- mat_aligned[sig_aligned != 0, , drop = FALSE]





mat <- as.matrix(logCPM_fit_bins_Filt)          # genes x bins
mat_scaled <- t(scale(t(mat)))               # gene-wise z-score across bins (shape-only)
mat_scaled[is.na(mat_scaled)] <- 0           # if any constant genes slipped through

library(clusterExperiment)

In [ ]:
%%R -o logCPM_fitDF

logCPM_fitDF <- as.data.frame(logCPM_fit)
colnames(logCPM_fitDF) <- colnames(y)

In [ ]:
Results = pd.concat([tab, shrinkedSIgnificance], axis = 1)
Results = Results[Results["X3-X2-X1"] == 1].copy()

Results

from scipy.stats import zscore


logCPM_fitDF_Filt = logCPM_fit_bins.loc[Results.index]
logCPM_fitDF_Filt = logCPM_fitDF_Filt

FittedFilt = pd.DataFrame(zscore(logCPM_fitDF_Filt.T).T, index=logCPM_fitDF_Filt.index, columns=logCPM_fitDF_Filt.columns)

adataGenes = ad.AnnData(FittedFilt)
adataGenes.var_names = [np.round(np.float32(i)) for i in adataGenes.var_names.tolist()]

sc.pp.neighbors(adataGenes, n_neighbors=20, use_rep="X")

adataGenes

leiden_kwargs = {'flavor': 'igraph', 'n_iterations': 2, 'directed': False}
# Override with user-provided kwargs if any

sc.tl.leiden(adataGenes, **leiden_kwargs, resolution=.1)

adataGenes.var

In [ ]:

cluster_labels = adataGenes.obs["leiden"].unique()
clusters = adataGenes.obs["leiden"]
trends = adataGenes.to_df()


n_rows = int(np.ceil(len(cluster_labels) / 3))
fig = plt.figure(figsize=[5.5 * 3, 2.5 * n_rows])

# Plot each cluster
for i, c in enumerate(cluster_labels):
    ax = fig.add_subplot(n_rows, 3, i + 1)
    cluster_trends = trends.loc[clusters.index[clusters == c], :]
    means = cluster_trends.mean()
    std = cluster_trends.std()

    ax.plot(
        cluster_trends.columns,
        cluster_trends.T,
        linewidth=0.5,color="lightgrey",
    )
    ax.plot(means.index, means)
    ax.plot(
        means.index,
        means - std,
        linestyle="--",
        linewidth=0.75,
    )
    ax.plot(
        means.index,
        means + std,
        linestyle="--",
        linewidth=0.75,
    )

    ax.set_title(f"Cluster {c}", fontsize=12)
    ax.tick_params("both", length=2, width=1, which="major")
    ax.tick_params(axis="both", which="major", labelsize=8, direction="in")


In [ ]:
import numpy as np

for i in adataGenes.obs["leiden"].unique():
    mask = adataGenes.obs["leiden"] == i
    sub = adataGenes[mask]

    X = sub.X
    if not isinstance(X, np.ndarray):
        X = X.toarray()

    centroid = X.mean(axis=0)
    dists = ((X - centroid) ** 2).sum(axis=1)
    hub_idx_sub = np.argmin(dists)

    hub_gene_name = sub.obs_names[hub_idx_sub]

    col = f"cluster{i}_hub"
    adataGenes.obs[col] = False
    adataGenes.obs.loc[hub_gene_name, col] = True

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cluster_labels = adataGenes.obs["leiden"].unique()
clusters = adataGenes.obs["leiden"]
trends = adataGenes.to_df()   # rows = genes, columns = samples / pseudotime / conditions

n_cols = 2
n_rows = int(np.ceil(len(cluster_labels) / n_cols))
fig = plt.figure(figsize=(5.5 * n_cols, 2.8 * n_rows))

top_n_hubs = 5

for i, c in enumerate(cluster_labels):
    ax = fig.add_subplot(n_rows, n_cols, i + 1)

    # genes in this cluster
    cluster_genes = clusters.index[clusters == c]
    cluster_trends = trends.loc[cluster_genes, :]

    # mean and std across genes
    means = cluster_trends.mean(axis=0)
    std = cluster_trends.std(axis=0)

    # plot all genes in grey
    ax.plot(
        cluster_trends.columns,
        cluster_trends.T,
        linewidth=0.5,
        color="lightgrey",
        alpha=0.8,
    )

    # plot cluster mean
    ax.plot(
        means.index,
        means.values,
        linewidth=1.5,
        color="black",
        label="cluster mean",
    )

    # plot mean ± std
    ax.plot(
        means.index,
        (means - std).values,
        linestyle="--",
        linewidth=0.75,
        color="black",
        alpha=0.7,
    )
    ax.plot(
        means.index,
        (means + std).values,
        linestyle="--",
        linewidth=0.75,
        color="black",
        alpha=0.7,
    )

    # get top n hub genes for this cluster
    sub = adataGenes[adataGenes.obs["leiden"] == c]

    X = sub.X
    if not isinstance(X, np.ndarray):
        X = X.toarray()

    centroid = X.mean(axis=0)
    dists = ((X - centroid) ** 2).sum(axis=1)

    # rank genes by closeness to centroid
    order = np.argsort(dists)
    top_idx = order[:min(top_n_hubs, len(order))]
    hub_genes = list(sub.obs_names[top_idx])

    # plot top hub trends
    for j, hub_gene in enumerate(hub_genes):
        hub_trend = trends.loc[hub_gene, :]
        ax.plot(
            hub_trend.index,
            hub_trend.values,
            linewidth=2.0 if j == 0 else 1.3,
            alpha=1.0 if j == 0 else 0.9,
            label=hub_gene,
        )

    # write hub gene names inside panel
    ax.text(
        0.02, 0.98,
        "top hubs:\n" + "\n".join(hub_genes),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.8),
    )

    ax.set_title(f"Cluster {c}", fontsize=12)
    ax.tick_params("both", length=2, width=1, which="major")
    ax.tick_params(axis="both", which="major", labelsize=8, direction="in")

plt.tight_layout()
plt.show()

# Store fitting, stats and clustering for export

In [ ]:
exportDF = adataGenes.to_df()
exportDF.columns = [f"fittedExpr_at_{i}um" for i in exportDF.columns]

exportDF = pd.concat([exportDF, Results.loc[exportDF.index]], axis = 1)
exportDF["GeneCluster"] = adataGenes.obs["leiden"]


saveDir = f"TrendsGenes_{nb_name}"
os.makedirs(saveDir, exist_ok=True)


exportDF.to_excel(os.path.join(saveDir, f"TrendsGenes_{DS}.xlsx"), index=True)